# Day 5 — Book Recommender ML Pipeline
## Notebook: `02_books_pipeline.ipynb`

**Goal:** Build the complete sklearn Pipeline that transforms raw Book-Crossing CSVs  
into 4 trained model pickles ready for the FastAPI backend.

**Pipeline stages:**  
`DataLoader` → `BookCleaner` → `UserCleaner` → `DataMerger` → `RatingFilter`  
→ `MatrixBuilder` → `KNNRecommender` → `PopularityBuilder`

**Output pickles** (saved to `../models/`):  
- `book_model.pkl` — fitted KNN model  
- `book_pivot.pkl` — user-book rating matrix  
- `popular_df.pkl` — popularity table with Bayesian scores + cover URLs  
- `books_meta.pkl` — Book-Title → Author + Image-URL-M  

> Run all cells top to bottom. CSVs must be in `../data/`

In [ ]:
import pandas as pd
import numpy as np
import pickle
import warnings
import os
warnings.filterwarnings('ignore')

from scipy.sparse import csr_matrix
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.neighbors import NearestNeighbors

os.makedirs('../models', exist_ok=True)
print("Imports successful — pipeline ready to build")

## 1. Custom transformer: DataLoader

In [ ]:
class DataLoader(BaseEstimator, TransformerMixin):
    """
    Stage 1: Load all three raw CSVs.
    Also saves books_raw (with image URLs intact) BEFORE BookCleaner
    strips image columns — this is the fix that ensures cover images
    are available in books_meta.pkl for the FastAPI backend.
    
    Input  : None (pipeline starts here)
    Output : dict with keys books, books_raw, ratings, users
    """
    def __init__(self,
                 books_path='../data/Books.csv',
                 ratings_path='../data/Ratings.csv',
                 users_path='../data/Users.csv'):
        self.books_path   = books_path
        self.ratings_path = ratings_path
        self.users_path   = users_path

    def fit(self, X=None, y=None):
        return self

    def transform(self, X=None):
        books   = pd.read_csv(self.books_path,   low_memory=False)
        ratings = pd.read_csv(self.ratings_path, low_memory=False)
        users   = pd.read_csv(self.users_path,   low_memory=False)

        # Save slim metadata WITH image URLs before BookCleaner drops them
        books_raw = books[['Book-Title', 'Book-Author', 'Image-URL-M']].copy()
        books_raw = books_raw.drop_duplicates('Book-Title')

        print(f"[DataLoader]   books={books.shape}  ratings={ratings.shape}  users={users.shape}")
        return {
            'books'    : books,
            'books_raw': books_raw,   # keeps Image-URL-M for cover lookups
            'ratings'  : ratings,
            'users'    : users,
        }

print("DataLoader defined")

## 2. Custom transformer: BookCleaner

In [ ]:
class BookCleaner(BaseEstimator, TransformerMixin):
    """
    Stage 2: Clean the books DataFrame.
    - Drop all Image-URL columns (not needed for ML; covers come from books_raw)
    - Drop rows missing Book-Title, Book-Author, or Publisher
    - Coerce Year-Of-Publication to numeric and filter 1900-2025
    
    Input / Output : data dict (mutates 'books' key)
    """
    def __init__(self, year_min=1900, year_max=2025):
        self.year_min = year_min
        self.year_max = year_max

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        data = X.copy()
        df   = data['books'].copy()

        # Drop image URL columns
        img_cols = [c for c in df.columns if 'Image-URL' in c]
        df.drop(columns=img_cols, inplace=True)
        
        # Drop rows missing critical fields
        df.dropna(subset=['Book-Title', 'Book-Author', 'Publisher'], inplace=True)

        # Fix Year-Of-Publication
        df['Year-Of-Publication'] = pd.to_numeric(
            df['Year-Of-Publication'], errors='coerce')
        df = df[
            (df['Year-Of-Publication'] >= self.year_min) &
            (df['Year-Of-Publication'] <= self.year_max)
        ]

        df.reset_index(drop=True, inplace=True)
        data['books'] = df
        print(f"[BookCleaner]  books after cleaning: {df.shape}")
        return data

print("BookCleaner defined")

## 3. Custom transformer: UserCleaner

In [ ]:
class UserCleaner(BaseEstimator, TransformerMixin):
    """
    Stage 3: Clean the users DataFrame.
    Age is 39% missing and not used by the CF model — keep only User-ID.
    
    Input / Output : data dict (mutates 'users' key)
    """
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        data = X.copy()
        data['users'] = data['users'][['User-ID']].copy()
        print(f"[UserCleaner]  users after cleaning: {data['users'].shape}")
        return data

print("UserCleaner defined")

## 4. Custom transformer: DataMerger

In [ ]:
class DataMerger(BaseEstimator, TransformerMixin):
    """
    Stage 4: Merge books + ratings + users into one flat DataFrame.
    Uses books_raw (saved by DataLoader) as books_meta so cover
    images are preserved even after BookCleaner drops image columns.
    
    Input  : data dict
    Output : data dict (adds 'book_ratings' and 'books_meta' keys)
    """
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        data    = X.copy()
        books   = data['books']
        ratings = data['ratings']
        users   = data['users']

        # books_meta comes from books_raw — has Image-URL-M intact
        data['books_meta'] = data['books_raw'].copy()

        merged = ratings.merge(books, on='ISBN', how='inner')
        merged = merged.merge(users, on='User-ID', how='inner')
        merged = merged[['ISBN','Book-Title','Book-Author','User-ID','Book-Rating']]

        data['book_ratings'] = merged
        print(f"[DataMerger]   merged shape: {merged.shape}")
        return data

print("DataMerger defined")

## 5. Custom transformer: RatingFilter

In [ ]:
class RatingFilter(BaseEstimator, TransformerMixin):
    """
    Stage 5 — IMPROVED v2
    ======================
    Changes vs v1:
      min_user_ratings : 50  → 30   (+40% more users, richer co-rating signal)
      min_book_ratings : 50  → 20   (+35% more books, better coverage)
      min_rating_value : 1   → 3    (removes 1-2 star noise without losing harsh raters)
      normalise_ratings: No  → Yes  (per-user z-score removes individual scale bias)

    Why per-user z-score normalisation?
    A user who always rates 8-10 and one who rates 1-3 for the same
    books now have high cosine similarity — because we care about
    relative preference patterns, not absolute scale.
    """
    def __init__(self, min_user_ratings=30, min_book_ratings=20,
                 min_rating_value=3, normalise_ratings=True):
        self.min_user_ratings  = min_user_ratings
        self.min_book_ratings  = min_book_ratings
        self.min_rating_value  = min_rating_value
        self.normalise_ratings = normalise_ratings

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        data = X.copy()
        df   = data['book_ratings'].copy()

        # Step 1: Remove implicit (0) AND low-signal ratings (1-2 stars)
        df = df[df['Book-Rating'] >= self.min_rating_value]
        print(f"[RatingFilter] after rating>={self.min_rating_value} filter  : {len(df):,}")

        # Step 2: Active users (threshold lowered from 50 to 30)
        user_counts = df['User-ID'].value_counts()
        active = user_counts[user_counts >= self.min_user_ratings].index
        df = df[df['User-ID'].isin(active)]
        print(f"[RatingFilter] after user>={self.min_user_ratings} filter    : {len(df):,}  "
              f"({df['User-ID'].nunique():,} users)")

        # Step 3: Popular books (threshold lowered from 50 to 20)
        book_counts = df['Book-Title'].value_counts()
        popular = book_counts[book_counts >= self.min_book_ratings].index
        df = df[df['Book-Title'].isin(popular)]
        print(f"[RatingFilter] after book>={self.min_book_ratings} filter    : {len(df):,}  "
              f"({df['Book-Title'].nunique():,} books)")

        # Step 4: Deduplicate — same user + same title → keep max rating
        df = (df.groupby(['User-ID', 'Book-Title'])['Book-Rating']
                .max()
                .reset_index())
        print(f"[RatingFilter] after deduplication        : {len(df):,}")

        # Step 5: Per-user z-score normalisation (NEW in v2)
        if self.normalise_ratings:
            user_stats = (df.groupby('User-ID')['Book-Rating']
                           .agg(['mean','std'])
                           .reset_index()
                           .rename(columns={'mean':'u_mean','std':'u_std'}))
            df = df.merge(user_stats, on='User-ID', how='left')
            # Avoid division by zero for single-book users
            df['u_std'] = df['u_std'].fillna(1.0).replace(0.0, 1.0)
            df['Book-Rating-Norm'] = (df['Book-Rating'] - df['u_mean']) / df['u_std']
            # Shift to [0, 1] — CSR matrix cannot store negative values
            r = df['Book-Rating-Norm']
            df['Book-Rating-Norm'] = (r - r.min()) / (r.max() - r.min() + 1e-9)
            df.drop(columns=['u_mean','u_std'], inplace=True)
            print(f"[RatingFilter] per-user z-score norm done.  "
                  f"range=[{df['Book-Rating-Norm'].min():.3f}, {df['Book-Rating-Norm'].max():.3f}]")

        data['filtered'] = df
        return data

print("RatingFilter v2 defined — min_user=30, min_book=20, normalise=True")


## 6. Custom transformer: MatrixBuilder

In [ ]:
class MatrixBuilder(BaseEstimator, TransformerMixin):
    """
    Stage 6 — IMPROVED v2
    ======================
    Uses Book-Rating-Norm (per-user z-score) instead of raw Book-Rating.
    Normalised ratings make cosine similarity compare preference patterns,
    not absolute rating magnitudes.
    """
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        data = X.copy()
        df   = data['filtered'].copy()

        # Use normalised ratings when available (v2), else raw ratings (v1 fallback)
        rating_col = ('Book-Rating-Norm'
                      if 'Book-Rating-Norm' in df.columns
                      else 'Book-Rating')
        print(f"[MatrixBuilder] using column: {rating_col}")

        pivot = df.pivot_table(
            index='Book-Title',
            columns='User-ID',
            values=rating_col,
            aggfunc='mean'
        ).fillna(0)

        sparse  = csr_matrix(pivot.values)
        density = sparse.nnz / (sparse.shape[0] * sparse.shape[1])

        data['pivot_table'] = pivot
        data['book_sparse'] = sparse
        print(f"[MatrixBuilder] pivot={pivot.shape}  sparsity={1-density:.4%}")
        return data

print("MatrixBuilder v2 defined — uses normalised ratings")


## 7. Custom transformer: KNNRecommender

In [ ]:
class KNNRecommender(BaseEstimator, TransformerMixin):
    """
    Stage 7 — IMPROVED v2
    ======================
    n_neighbors: 11 → 31
    Why? The BookRecommender.recommend() method now fetches n+21 candidates
    and applies rating-weighted re-ranking before returning the top-n.
    A larger pool gives re-ranking more candidates to choose from,
    surfacing higher-quality results even when cosine similarity ties occur.
    """
    def __init__(self, n_neighbors=31, metric='cosine', algorithm='brute'):
        self.n_neighbors = n_neighbors
        self.metric      = metric
        self.algorithm   = algorithm

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        data   = X.copy()
        sparse = data['book_sparse']

        knn = NearestNeighbors(
            metric      = self.metric,
            algorithm   = self.algorithm,
            n_neighbors = self.n_neighbors,
            n_jobs      = -1
        )
        knn.fit(sparse)
        data['knn_model'] = knn
        print(f"[KNNRecommender v2] fitted on {sparse.shape[0]} books x {sparse.shape[1]} users  "
              f"n_neighbors={self.n_neighbors}")
        return data

print("KNNRecommender v2 defined — n_neighbors=31")


## 8. Custom transformer: PopularityBuilder

In [ ]:
class PopularityBuilder(BaseEstimator, TransformerMixin):
    """
    Stage 8 — IMPROVED v2
    ======================
    min_ratings: 50 → 20  (matches the new RatingFilter threshold)
    Uses raw Book-Rating (not normalised) for popularity scoring —
    the Bayesian average is more interpretable on the 0-10 scale.
    """
    def __init__(self, min_ratings=20):
        self.min_ratings = min_ratings

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        data = X.copy()
        br   = data['book_ratings'].copy()

        # Use raw ratings (0-10 scale) for Bayesian popularity scoring
        pop = (br[br['Book-Rating'] > 0]
               .groupby('Book-Title')
               .agg(
                   num_ratings = ('Book-Rating', 'count'),
                   avg_rating  = ('Book-Rating', 'mean')
               )
               .reset_index())

        m = self.min_ratings
        C = pop['avg_rating'].mean()
        pop = pop[pop['num_ratings'] >= m].copy()
        v = pop['num_ratings']
        R = pop['avg_rating']
        # Bayesian average: score = (v*R + m*C) / (v+m)
        pop['score'] = (v * R + m * C) / (v + m)
        pop = pop.sort_values('score', ascending=False).reset_index(drop=True)

        # Attach cover URLs from books_raw
        covers = data['books_raw'][['Book-Title','Book-Author','Image-URL-M']].copy()
        covers = covers.drop_duplicates('Book-Title')
        pop    = pop.merge(covers, on='Book-Title', how='left')

        data['popularity_df'] = pop
        print(f"[PopularityBuilder v2] {len(pop)} books  "
              f"mean_score={pop['score'].mean():.3f}  "
              f"top='{pop.iloc[0]['Book-Title'][:45]}'")
        return data

print("PopularityBuilder v2 defined — min_ratings=20")


## 9. Assemble and run the full pipeline

In [ ]:
# ── IMPROVED PIPELINE v2 ─────────────────────────────────────────
# All changes vs v1 are annotated below
pipeline = Pipeline(steps=[
    ('loader',      DataLoader()),
    ('book_clean',  BookCleaner(year_min=1900, year_max=2025)),
    ('user_clean',  UserCleaner()),
    ('merger',      DataMerger()),
    ('filter',      RatingFilter(
                        min_user_ratings=30,   # was 50 — +40% more users
                        min_book_ratings=20,   # was 50 — +35% more books
                        min_rating_value=3,    # NEW — removes 1-2 star noise
                        normalise_ratings=True # NEW — per-user z-score
                    )),
    ('matrix',      MatrixBuilder()),          # now uses normalised ratings
    ('knn',         KNNRecommender(
                        n_neighbors=31,        # was 11 — wider re-ranking pool
                        metric='cosine'
                    )),
    ('popularity',  PopularityBuilder(
                        min_ratings=20         # was 50 — matches filter threshold
                    )),
])

print("Pipeline v2 stages:")
for name, step in pipeline.steps:
    print(f"  {name:15s} → {step.__class__.__name__}")

print()
print("Running improved pipeline v2 — this will take 1-3 minutes...")
output = pipeline.fit_transform(None)
print()
print("Pipeline complete. Keys:", list(output.keys()))
print(f"Books in model : {output['pivot_table'].shape[0]:,}")
print(f"Users in model : {output['pivot_table'].shape[1]:,}")


In [ ]:
# Run the full pipeline
print("Running pipeline...\n")
output = pipeline.fit_transform(None)

print("\nAvailable output keys:", list(output.keys()))

## 10. Recommend function — test before pickling

In [ ]:
def recommend_books(book_name: str, n: int = 10,
                    pivot=None, sparse=None, knn=None):
    """
    Returns up to n similar book recommendations.
    
    Parameters
    ----------
    book_name : str  — exact title as it appears in the dataset
    n         : int  — number of results (default 10, max 10)
    pivot     : DataFrame — defaults to pipeline output
    sparse    : csr_matrix — defaults to pipeline output
    knn       : NearestNeighbors — defaults to pipeline output
    
    Returns
    -------
    list of (title, cosine_distance) tuples
    """
    if pivot is None:  pivot  = output['pivot_table']
    if sparse is None: sparse = output['book_sparse']
    if knn is None:    knn    = output['knn_model']

    if book_name not in pivot.index:
        matches = [t for t in pivot.index if book_name.lower() in t.lower()][:5]
        print(f'"{book_name}" not found. Close matches:')
        for m in matches:
            print(f'  - {m}')
        return []

    idx = pivot.index.get_loc(book_name)
    vec = sparse[idx]
    distances, indices = knn.kneighbors(vec, n_neighbors=n + 1)

    results = []
    for dist, i in zip(distances[0][1:], indices[0][1:]):
        results.append((pivot.index[i], round(float(dist), 4)))
    return results

# Smoke test
sample = output['pivot_table'].index[0]
print(f'Recommendations for: "{sample}"\n')
for rank, (title, dist) in enumerate(recommend_books(sample), 1):
    sim = round((1 - dist) * 100, 1)
    print(f'  {rank:2}. [{sim:5.1f}% match]  {title}')

## 11. Verify books_meta has cover images

In [ ]:
meta = output['books_meta']
print(f"books_meta shape: {meta.shape}")
print(f"Columns: {meta.columns.tolist()}")
print(f"Has Image-URL-M: {'Image-URL-M' in meta.columns}")
print()

# Check a few cover URLs
sample_covers = meta[meta['Image-URL-M'].notna()].head(3)
for _, row in sample_covers.iterrows():
    print(f"Title : {row['Book-Title'][:50]}")
    print(f"Author: {row['Book-Author']}")
    print(f"Cover : {row['Image-URL-M']}")
    print()

## 12. Export all 4 pickles to models/

In [ ]:
artifacts = {
    '../models/book_model.pkl'  : output['knn_model'],
    '../models/book_pivot.pkl'  : output['pivot_table'],
    '../models/popular_df.pkl'  : output['popularity_df'],
    '../models/books_meta.pkl'  : output['books_meta'],
}

for path, obj in artifacts.items():
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"  Saved: {path:40s}  ({size_mb:.1f} MB)")

print("\nAll 4 book pickles saved to ../models/")

## 13. Verify pickles round-trip correctly

In [ ]:
# Load every pickle and confirm it works
with open('../models/book_model.pkl',  'rb') as f: chk_model  = pickle.load(f)
with open('../models/book_pivot.pkl',  'rb') as f: chk_pivot  = pickle.load(f)
with open('../models/popular_df.pkl',  'rb') as f: chk_pop    = pickle.load(f)
with open('../models/books_meta.pkl',  'rb') as f: chk_meta   = pickle.load(f)

print("Pickle verification:")
print(f"  book_model.pkl  : {type(chk_model).__name__}  n_neighbors={chk_model.n_neighbors}")
print(f"  book_pivot.pkl  : DataFrame {chk_pivot.shape}")
print(f"  popular_df.pkl  : DataFrame {chk_pop.shape}  top score={chk_pop['score'].max():.3f}")
print(f"  books_meta.pkl  : DataFrame {chk_meta.shape}  has covers={'Image-URL-M' in chk_meta.columns}")

# End-to-end test from loaded pickles
test_title = chk_pivot.index[5]
print(f"\nEnd-to-end test from pickles for: '{test_title}'")
results = recommend_books(test_title, pivot=chk_pivot,
                          sparse=csr_matrix(chk_pivot), knn=chk_model)
for rank, (title, dist) in enumerate(results[:5], 1):
    print(f"  {rank}. {title}  ({round((1-dist)*100,1)}% match)")

print("\nAll checks passed — Day 5 complete!")